In [1]:
import pandas as pd
from neo4j import GraphDatabase
from pathlib import Path
from getpass import getpass

In [2]:
project_folder = Path.cwd()

if project_folder.name == "notebooks":
    project_folder = project_folder.parent

output_folder = project_folder / "data" / "outputs"

junctions_df = pd.read_csv(
    output_folder / "junctions.csv",
    dtype={"id": str, "name": str}
)

roads_df = pd.read_csv(
    output_folder / "roads.csv",
    dtype={
        "id": str,
        "fromJunction": str,
        "toJunction": str
    }
)

print("Junctions:", len(junctions_df))
print("Roads:", len(roads_df))

Junctions: 276
Roads: 794


In [3]:
junctions_df["id"] = "SUMO_" + junctions_df["id"]
junctions_df["name"] = junctions_df["id"]

roads_df["id"] = "SUMO_" + roads_df["id"]

roads_df["fromJunction"] = "SUMO_" + roads_df["fromJunction"]
roads_df["toJunction"] = "SUMO_" + roads_df["toJunction"]

print("Identifiers prepared successfully!")

Identifiers prepared successfully!


In [4]:
uri = "bolt://localhost:7687"
username = "neo4j"

password = getpass("Enter Neo4j password: ")

driver = GraphDatabase.driver(
    uri,
    auth=(username, password)
)

driver.verify_connectivity()

print("Connected to Neo4j!")

Connected to Neo4j!


In [5]:
junction_data = junctions_df.to_dict("records")

query = """
UNWIND $batch AS row

MERGE (j:Junction {id: row.id})

SET j.name = row.name,
    j.junctionId = row.id,
    j.x = row.x,
    j.y = row.y
"""

batch_size = 500

with driver.session(database="neo4j") as session:

    for i in range(0, len(junction_data), batch_size):

        batch = junction_data[i:i + batch_size]

        session.run(query, batch=batch).consume()

print("Junctions imported successfully!")

Junctions imported successfully!


In [6]:
road_data = roads_df.to_dict("records")

query = """
UNWIND $batch AS row

MERGE (r:Road {id: row.id})

SET r.roadName = row.roadName,
    r.length = row.length,
    r.speedLimit = row.speedLimit,
    r.lanes = row.lanes,
    r.status = row.status
"""

with driver.session(database="neo4j") as session:

    for i in range(0, len(road_data), batch_size):

        batch = road_data[i:i + batch_size]

        session.run(query, batch=batch).consume()

print("Roads imported successfully!")

Roads imported successfully!


In [7]:
query = """
UNWIND $batch AS row

MATCH (r:Road {id: row.id})
MATCH (a:Junction {id: row.fromJunction})
MATCH (b:Junction {id: row.toJunction})

MERGE (r)-[:CONNECTED_TO]->(a)
MERGE (r)-[:CONNECTED_TO]->(b)
"""

with driver.session(database="neo4j") as session:

    for i in range(0, len(road_data), batch_size):

        batch = road_data[i:i + batch_size]

        session.run(query, batch=batch).consume()

print("Road-junction relationships created!")

Road-junction relationships created!


In [8]:
driver.close()

print("Neo4j connection closed successfully!")

Neo4j connection closed successfully!
